# Sound Processing Lab - Part #1

In this notebook you'll learn:
- How to create a sound
- How to load a sound file and play it
- Prints basic metadata (sampling rate, duration, samples, channels)
- Oscillogram (waveform) full and zoomed
- Spectrogram (STFT, dB)
- Amplitude and Frequency histograms
- Build and apply filters on a sound

In [ ]:
# Imports and configuration
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from tabulate import tabulate
import sounddevice as sd

# Generate a sound wave

Choose `frequency`, `sampling rate` and `duration`.
Then modify `xlim` to watch the scale of the x axis and see full cycles.

NOTE: FOR THE SAKE OF EVERYBODY IN THE ROOM, DO NOT PLAY FREQUENCIES OVER 1000 Hz (to preserve yours and others' hearing; also please keep volume low. Thank you!)

In [ ]:
# Signal parameters
frequency = 500          # Frequency in Hz (1000 cycles per second)
sampling_rate = 44100     # Standard audio sampling rate (44.1 kHz)
duration = 1              # in seconds

# Generate time axis and the sine wave signal
time_axis = np.linspace(0, duration, int(sampling_rate * duration), endpoint=False)
sine_signal = np.sin(2 * np.pi * frequency * time_axis)

# Plot the generated sine wave
plt.figure(figsize=(10, 4))
plt.plot(time_axis * 1000, sine_signal, color='blue', linewidth=2)  # time converted to ms
plt.title(f'Pure {frequency} Hz Sine Wave', fontsize=14)
plt.xlabel('Time (ms)', fontsize=12)
plt.ylabel('Amplitude', fontsize=12)
plt.grid(True, linestyle='--')
plt.xlim(0, duration * 5)
plt.tight_layout()
plt.show()

# Play the audio signal
print(f"Playing {frequency} Hz tone...")
sd.play(sine_signal, sampling_rate)
sd.wait()  # Wait until the audio finishes playing
print("Playback finished.")

# Now open, play and extract basic metadata of an MP3

- How to load a sound file and play it
- Prints basic metadata (sampling rate, duration, samples, channels)

In [ ]:
plt.rcParams['figure.figsize'] = (12, 4)

# Path to your audio file: change this to the MP3 you downloaded
audio_path = '..\\sounds\\dog\\dog_bark_1.mp3'  # <-- REPLACE with your file path (mp3 is supported)

if not os.path.exists(audio_path):
    print(f'File not found: {audio_path} — please update the path and re-run this cell')
else:
    ext = os.path.splitext(audio_path)[1].lower()
    y = None
    sr = None
    # If pydub is available, use it to load reliably (handles mp3 via ffmpeg)
    try:
        y, sr = librosa.load(audio_path, sr=None, mono=False)
    except Exception as e:
        print('pydub not available or failed to read the file. Error:', e)

    # librosa returns shape (n,) for mono or (channels, n) for stereo when mono=False
    if y is None:
        print('Failed to load audio.')
    else:
        if hasattr(y, 'ndim') and y.ndim == 2:
            n_channels = y.shape[0]
            n_samples = y.shape[1]
        else:
            n_channels = 1
            n_samples = y.shape[0]
        duration_sec = n_samples / float(sr)

        print('Loaded:', audio_path)
        print('Sampling rate (Hz):', sr)
        print('Channels:', n_channels)
        print('Total samples:', n_samples)
        print(f'Duration (s): {duration_sec:.3f}')
        print(f'Dataset size:  {y.size} - Dataset shape: {y.shape}')  # total samples and shape (channels, samples) or (samples,)

        # create a mono version for plotting/analysis (average channels if stereo)
        if n_channels > 1:
            y_mono = librosa.to_mono(y)
        else:
            y_mono = y

        # keep these variables in the notebook namespace for later cells
        globals()['y'] = y
        globals()['y_mono'] = y_mono
        globals()['sr'] = sr
        globals()['audio_path'] = audio_path

        # Play audio inline
        display(Audio(y_mono, rate=sr))

# Observe the anatomy of a sound wave dataset
You will find one row per channel, and one column per sample present in the given sound (yes, per sample).

In [ ]:
channel_matrix = y if y.ndim > 1 else [y]      
# Creates the structure of the table with channel labels and sample values (first 20 samples for brevity)
table = [[f"Channel {i + 1}"] + list(channel[:20]) for i, channel in enumerate(channel_matrix)]
headers = ["Audio"] + [f"Smp {i+1}" for i in range(20)]
print(tabulate(table, headers=headers, tablefmt="table"))

## Oscillogram (Waveform) — full and zoomed

The first plot shows the full waveform; the second plot zooms into a short window so you can inspect individual cycles. Try changing `zoom_start_sec` and `zoom_duration_sec`.

In [ ]:
# Oscillogram plots
try:
    # full waveform
    plt.figure()
    librosa.display.waveshow(y_mono, sr=sr)
    plt.title('Oscillogram — Full waveform')
    plt.xlabel('Time (s)')
    plt.tight_layout()
    plt.show()

    # zoom parameters (seconds)
    zoom_start_sec = 0.5  # change to inspect a different region
    zoom_duration_sec = 0.05  # short window to see cycles
    start_sample = int(zoom_start_sec * sr)
    end_sample = int((zoom_start_sec + zoom_duration_sec) * sr)

    plt.figure()
    time_axis = np.linspace(zoom_start_sec, zoom_start_sec + zoom_duration_sec, end_sample - start_sample)
    plt.plot(time_axis, y_mono[start_sample:end_sample])
    plt.title(f'Oscillogram — Zoomed ({zoom_duration_sec}s starting at {zoom_start_sec}s)')
    plt.xlabel('Time (s)')
    plt.tight_layout()
    plt.show()
except NameError:
    print('Run the previous cell to load the audio (check audio_path).')

## Spectrogram — Logaritmic Frequency Scale
Try changing `xlim` and `ylim` to see the time/frequency resolution tradeoff.

In [ ]:
# Spectrogram
plt.figure(figsize=(12, 6))

librosa.display.specshow(librosa.amplitude_to_db(np.abs(librosa.stft(y_mono)), ref=np.max),
                         sr=sr,
                         x_axis='time',
                         y_axis='log', # Uses logarithmic frequency scale for better visibility of lower frequencies (like our own hearing)
                         cmap='magma') # Explore (magma, viridis)

plt.title('Spectrogram (dB, Logaritmic Scale)', fontsize=16)
plt.colorbar(format='%+2.0f dB') # Intensity colorbar in decibels
plt.ylabel('Frequency (Logarithmic Scale)', fontsize=12)
plt.tight_layout()
plt.xlim(0, 5.424) # Limit to first 5 seconds for better visibility (adjust as needed)  / set to None for full range
#plt.xlim(0, None)
plt.ylim(20, sr/2) # Limit frequency range to human hearing (20 Hz to Nyquist frequency) / set to None for full range
#plt.ylim(20, None)
plt.show()

## Amplitude and Frequencies Histogram

Plot a histogram of sample amplitudes and frequencies to inspect distribution and clipping.

In [ ]:
# Histogram of amplitudes
try:
    samples = y_mono.flatten()
    plt.figure()
    plt.hist(samples, bins=100, color='gray')
    plt.title('Histogram of sample amplitudes')
    plt.xlabel('Amplitude')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.xlim(-1, 1) # Audio samples are typically in the range [-1, 1]
    plt.show()

    # basic stats
    print('Min amplitude:', samples.min())
    print('Max amplitude:', samples.max())
    print('Mean:', samples.mean())
    print('RMS:', np.sqrt(np.mean(samples**2)))
except NameError:
    print('Run the audio loading cell first (check audio_path).')

# Histogram of frequencies
try:
    # Compute the Fast Fourier Transform (FFT) and corresponding frequencies
    samples = y_mono.flatten()
    fft_data = np.fft.rfft(samples)
    frequencies = np.fft.rfftfreq(len(samples), d=1/sr)
    magnitudes = np.abs(fft_data)

    plt.figure()
    # Use weights to make the histogram represent the energy distribution across frequencies
    plt.hist(frequencies, bins=100, weights=magnitudes, color='purple')
    plt.title('Histogram of frequencies (Energy Distribution)')
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Accumulated Magnitude')
    plt.tight_layout()
    plt.show()

    # basic stats
    print('Dominant frequency:', frequencies[np.argmax(magnitudes)], 'Hz')
    print('Min magnitude:', magnitudes.min())
    print('Max magnitude:', magnitudes.max())
    print('Mean magnitude:', magnitudes.mean())
except NameError:
    print('Run the audio loading cell first (check y_mono and sample_rate).')

## What have we learned?

Let's discuss in class.



Got more time and want to continue playing? Cool!
* Go to https://pixabay.com/ and pick a sound, place it under <root>\sounds and knock yourself out!
* Record your own voice and plot oscillograms, spectrograms and histograms; split words, zoom-in/out and try to find your own words and their responses in frequency and time domains.
    * Question: What happens when you pronounce oclusive or fricative consonants? When spelling words, which particles make you close your mouth or open it? Can you identify silences?